In [26]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [27]:
NEGATIVE_EMOTIONS = [
    "sadness", "grief", "fear", "anger", "remorse",
    "disappointment", "disapproval", "disgust",
    "annoyance", "nervousness", "embarrassment"
]  # custom order: high severity first

In [28]:
# Load dataset
from datasets import load_dataset

goemo_dataset = load_dataset("go_emotions")

# Load label names
label_list = goemo_dataset["train"].features["labels"].feature.names

# Map negative emotion name -> original index in GoEmotions
neg_emotion_to_original_id = {emo: label_list.index(emo) for emo in NEGATIVE_EMOTIONS}

# Map original id -> new compact id (0 to 10)
original_id_to_compact_id = {v: i for i, v in enumerate(sorted(neg_emotion_to_original_id.values()))}

# Step 2: Filter dataset to include only samples with exactly one negative emotion
def assign_single_label(example):
    neg_label_ids = [l for l in example["labels"] if l in original_id_to_compact_id]
    if len(neg_label_ids) == 1:
        original_label = neg_label_ids[0]
        example["label"] = original_id_to_compact_id[original_label]  # map to 0..10
    else:
        example["label"] = -1
    return example

filtered_dataset = goemo_dataset.map(assign_single_label)
# Filter out invalid
filtered_dataset = filtered_dataset.filter(lambda x: x["label"] != -1)

In [29]:
from transformers import BertTokenizerFast

# Step 4: Tokenize text
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = filtered_dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text", "labels", "id"])
# Step 4: Set format for PyTorch
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

dataset = tokenized_dataset["train"].train_test_split(test_size=0.2, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]

loading file vocab.txt from cache at /Users/bennyxiong/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/vocab.txt
loading file tokenizer.json from cache at /Users/bennyxiong/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /Users/bennyxiong/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/tokenizer_config.json
loading configuration file config.json from cache at /Users/bennyxiong/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,

In [ ]:
from transformers import BertForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import torch

# Model setup
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(NEGATIVE_EMOTIONS)
)

# Metrics
def compute_metrics(p):
    preds = torch.argmax(torch.tensor(p.predictions), axis=1)
    labels = p.label_ids
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

loading configuration file config.json from cache at /Users/bennyxiong/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8",
    "9": "LABEL_9",
    "10": "LABEL_10"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_10": 10,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "LABEL_6": 6,
    "LABEL_7": 7,
    "LABEL_8": 8,
    "LABEL_9": 9
  },
  "layer_norm_eps": 1e-12,
  "max_position_

In [31]:
trainer.train()

***** Running training *****
  Num examples = 6902
  Num Epochs = 5
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 2160
  Number of trainable parameters = 109490699





































































































                                                 

                                            
  3%|▎         | 63/2160 [04:40<53:20,  1.53s/it]


{'loss': 2.0684, 'learning_rate': 1.9074074074074075e-05, 'epoch': 0.23}






































































































                                                 

                                            
  3%|▎         | 63/2160 [07:12<53:20,  1.53s/it]


{'loss': 1.7184, 'learning_rate': 1.814814814814815e-05, 'epoch': 0.46}






































































































                                                 

                                            
  3%|▎         | 63/2160 [09:44<53:20,  1.53s/it]


...





























































***** Running Evaluation *****
  Num examples = 1726
  Batch size = 64






















































                                                   

                                               
                                       


  3%|▎         | 63/2160 [1:05:47<53:20,  1.53s/it]




Saving model checkpoint to ./results/checkpoint-2160
Configuration saved in ./results/checkpoint-2160/config.json


{'eval_loss': 1.3583561182022095, 'eval_accuracy': 0.5979142526071842, 'eval_f1': 0.5954591014498867, 'eval_precision': 0.594460410613401, 'eval_recall': 0.5979142526071842, 'eval_runtime': 46.1468, 'eval_samples_per_second': 37.402, 'eval_steps_per_second': 0.585, 'epoch': 5.0}


Model weights saved in ./results/checkpoint-2160/pytorch_model.bin


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./results/checkpoint-1296 (score: 0.597049681576671).
                                                   

                                               
100%|██████████| 2160/2160 [1:03:39<00:00,  1.77s/it]

{'train_runtime': 3819.8414, 'train_samples_per_second': 9.034, 'train_steps_per_second': 0.565, 'train_loss': 0.8982123163011339, 'epoch': 5.0}


TrainOutput(global_step=2160, training_loss=0.8982123163011339, metrics={'train_runtime': 3819.8414, 'train_samples_per_second': 9.034, 'train_steps_per_second': 0.565, 'train_loss': 0.8982123163011339, 'epoch': 5.0})

In [32]:
trainer.save_model("model")
tokenizer.save_pretrained("model")

Saving model checkpoint to model
Configuration saved in model/config.json
Model weights saved in model/pytorch_model.bin
tokenizer config file saved in model/tokenizer_config.json
Special tokens file saved in model/special_tokens_map.json


('model/tokenizer_config.json',
 'model/special_tokens_map.json',
 'model/vocab.txt',
 'model/added_tokens.json',
 'model/tokenizer.json')